# Prepare MLF1

Download and clean this ABS source, save its workbook and CSV, then publish its Delta table directly. Each notebook runs independently.

Use the job parameters described in [README](../README.md), and fill in `catalog` and `schema` in the final publishing cell.

Use only the underlying Data 1 sheet, not the presentation pivot. Keep the published ASGS geography and combined regions. Modelled identifies the source; it does not assert a seasonal-adjustment status.


## 1. Setup and source settings

Import the tools used by the notebook and set the dataset names, filenames, download address and workbook layout. Edit these settings when ABS changes this source, and review the workbook before updating the reference file.

In [ ]:
# Import the tools for dates, file paths, tables, downloads and notebook displays.
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests
from IPython.display import display

# Set the dataset name, filenames and ABS download address.
dataset = "MLF1"
dataset_key = "mlf1"
baseline_file = "MLF1.xlsx"
download_file = "MLF1.xlsx"
base_url = "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/"

# Select the underlying data sheet and the heading row (Excel row 4, counted from zero).
data_sheet = "Data 1"
mlf_header_row = 3

## 2. Job parameters and next release

The job supplies `run_date` (`{{job.start_time.iso_date}}`, the job start date in UTC), `job_run_id` (`{{job.run_id}}`) and `workspace_path` (the absolute `/Workspace/...` path to the project folder). MLF1 determines its release month automatically; it does not use the `release_month` job parameter.

Keep the reference workbook in `referencedatasets/`. The next release is one month after the later of its latest Data 1 month and the latest completed MLF1 release recorded in `outputs/mlf1/*/sources.csv`. A run is completed only after the CSV and Delta table have been written. Install pandas, openpyxl and requests in the task environment.

In [ ]:
# Create text inputs for the run date, job run ID and workspace project path.
dbutils.widgets.text("run_date", "", "Job run date (YYYY-MM-DD, UTC)")
dbutils.widgets.text("job_run_id", "", "Job run ID")
dbutils.widgets.text("workspace_path", "", "Workspace project path")

# Read the input values and convert the workspace path into a file path object.
run_date = dbutils.widgets.get("run_date")
job_run_id = dbutils.widgets.get("job_run_id")
workspace = Path(dbutils.widgets.get("workspace_path"))
# Check that the job run ID is filled in and is not an unresolved {{...}} placeholder.
assert job_run_id and "{{" not in job_run_id, "Supply the resolved job run ID"
# Check that the workspace project path is filled in.
assert dbutils.widgets.get("workspace_path"), "Supply workspace_path"

# Set the reference folder and the output folder named with the run date and job run ID.
baseline = workspace / "referencedatasets"
output = workspace / "outputs" / dataset_key
run = output / f"{run_date}_{job_run_id}"

# Read the Month column from the reference workbook’s Data 1 sheet and find its latest month.
baseline_dates = pd.read_excel(baseline / baseline_file, sheet_name=data_sheet, header=mlf_header_row, usecols=[0])
baseline_latest_month = pd.to_datetime(baseline_dates.iloc[:, 0]).max()
# Find the source records from completed MLF1 runs and read their release months.
completed_sources = output.glob("*/sources.csv")
processed_months = [pd.to_datetime(pd.read_csv(source_file)["reference_period"]).max() for source_file in completed_sources]

# Choose the next month after the latest reference or completed release, and format it as YYYY-MM.
download_month = max([baseline_latest_month, *processed_months]) + pd.offsets.MonthBegin(1)
release_month = download_month.strftime("%Y-%m")
# Display the dataset, selected release month and output folder.
print(dataset, release_month, run)

## 3. Download the release workbook

In [ ]:
# Build the download URL from the ABS address, selected release month and filename.
source_url = base_url + f"{download_month:%b-%Y}/".lower() + download_file
# Record the current time in UTC for the source record.
retrieved_at = datetime.now(timezone.utc).isoformat()
# Download the workbook and stop if the server reports an unsuccessful response.
response = requests.get(source_url, timeout=120, headers={"Cache-Control": "no-cache"})
response.raise_for_status()
# Check that the response starts with the ZIP file marker used by XLSX files.
assert response.content.startswith(b"PK"), f"{dataset}: download is not an XLSX file"

# Create the raw-data folder and save the downloaded workbook there.
(run / "raw").mkdir(parents=True, exist_ok=True)
(run / "raw" / download_file).write_bytes(response.content)

# Display the download URL.
print(source_url)

## 4. Read and validate the workbook

Read the reference and downloaded workbooks, then compare the sheets, table identity, headings and sex, age and region categories. Check for missing or non-numeric values, equal row counts across months and the selected release month.

In [ ]:
# Open the reference and downloaded workbooks and read the Data 1 sheet from each.
baseline_workbook = pd.ExcelFile(baseline / baseline_file)
downloaded_workbook = pd.ExcelFile(run / "raw" / download_file)
mlf_base = pd.read_excel(baseline_workbook, sheet_name=data_sheet, header=None)
mlf_raw = pd.read_excel(downloaded_workbook, sheet_name=data_sheet, header=None)

# Display the sheet names and check that both workbooks contain the same sheets and table title.
print("MLF1 sheets:", downloaded_workbook.sheet_names)
assert set(downloaded_workbook.sheet_names) == set(baseline_workbook.sheet_names), "MLF1: workbook sheets changed"
assert mlf_raw.iloc[1, 0] == mlf_base.iloc[1, 0], "MLF1: table identity changed"

# Read from the heading row onwards, excluding columns that are entirely empty in that range.
mlf_base = mlf_base.iloc[mlf_header_row:].dropna(axis=1, how="all")
mlf_raw = mlf_raw.iloc[mlf_header_row:].dropna(axis=1, how="all")
# Check that column names, units, order and geography version match the reference.
assert mlf_raw.iloc[0].tolist() == mlf_base.iloc[0].tolist(), "MLF1: column names, units, order or geography vintage changed"

# Select the rows below the headings and check that no dates, labels or estimates are missing.
values = mlf_raw.iloc[1:]
baseline_values = mlf_base.iloc[1:]
assert values.notna().all().all(), "MLF1: missing dimension or estimate"
# Check that all estimate columns contain numbers.
assert values.iloc[:, 4:].infer_objects().dtypes.map(pd.api.types.is_numeric_dtype).all(), "MLF1: non-numeric observations"
# Compare the sex, age and region categories with those in the reference workbook.
for column in range(1, 4):
    assert set(values.iloc[:, column]) == set(baseline_values.iloc[:, column]), f"MLF1: categories changed in {mlf_raw.iloc[0, column]}"

# Read the dates and count how many rows belong to each month.
dates = pd.to_datetime(values.iloc[:, 0])
rows_per_month = dates.value_counts().sort_index()
# Check that every month has the same number of sex, age and region rows.
assert rows_per_month.nunique() == 1, "MLF1: months have different numbers of sex × age × region rows"
# Check that the latest month matches the selected release month.
latest_month = dates.max()
assert latest_month == download_month, f"MLF1: latest month is not {download_month:%B %Y}"

# Display the number of data rows and columns, plus the latest monthly row counts.
print(values.shape)
display(rows_per_month.rename_axis("month").rename("rows").tail())

## 5. Reshape and clean the workbook

Apply the column headings, convert the dates, reshape estimates into one row per measure and extract the region codes and labels. Keep the published combined regions and label the source as Modelled.

In [ ]:
# Use the heading row as column names, reset row numbers and identify the column data types.
mlf_raw = mlf_raw.iloc[1:].set_axis(mlf_raw.iloc[0].tolist(), axis=1).reset_index(drop=True).infer_objects()
# Identify the sex, age, region and estimate columns, and convert Month to dates.
dimension_columns = mlf_raw.columns[1:4].tolist()
measure_columns = mlf_raw.columns[4:].tolist()
region_column = dimension_columns[2]
mlf_raw["Month"] = pd.to_datetime(mlf_raw["Month"])

# Rename the identifying columns and turn the estimate columns into measure and value rows.
mlf = mlf_raw.rename(columns={"Month": "date", "Sex": "sex", "Age": "age", region_column: "region"}).melt(
    id_vars=["date", "sex", "age", "region"], value_vars=measure_columns, var_name="measure"
)
# Split each region label into its three-digit code and name, keeping combined-region names together.
mlf[["region_code", "region"]] = mlf["region"].str.extract(r"^(\d{3}) (.+)$")
# Check that a three-digit region code was extracted for every row.
assert mlf["region_code"].notna().all(), "MLF1: region-code format changed"
# Remove the thousands unit from the measure names and store it in a separate column.
mlf["measure"] = mlf["measure"].str.replace(" ('000)", "", regex=False)
mlf["unit"] = "'000"
# Label every row as coming from the Modelled series.
mlf["series_type"] = "Modelled"
# Display the table size and the first five rows for review.
print(mlf.shape)
display(mlf[["date", "region_code", "region", "sex", "age", "measure", "value"]].head())

## 6. Save the cleaned CSV

Save the cleaned data under `outputs/mlf1/<run_date>_<job_run_id>/`. Runs without `sources.csv` are incomplete and do not advance the next release. Retrying an incomplete run with the same date and job run ID overwrites its raw workbook and CSV.

In [ ]:
# Select the output columns and rename the region code and name to SA4 labels.
df = mlf[["date", "measure", "sex", "age", "region_code", "region", "series_type", "unit", "value"]].rename(
    columns={"region_code": "sa4_code", "region": "sa4_region"}
)

# Name the CSV using the dataset and release month, then save it without row numbers and with dates as YYYY-MM-DD.
csv_file = run / f"{dataset_key}_{release_month}.csv"
df.to_csv(csv_file, index=False, date_format="%Y-%m-%d")
# Display the table size and the first five rows for review.
print(df.shape)
display(df.head())

## 7. Publish the Delta table

Replace `YOUR_CATALOG` and `YOUR_SCHEMA` with your existing destination names. `dataset_key` supplies this source's table name. Run this cell in Databricks after the CSV export.

The write creates a missing Delta table or overwrites its data. It uses the cleaned `df` already in memory; no staging Delta table or separate publisher is needed. This source updates independently of the other two. Plain overwrite does not automatically replace an existing table's schema.

After publishing, write `sources.csv` with this release's source details. Its presence tells the next run that this release completed successfully. A full rerun after completion selects the next release; retry only the publishing cell to republish the current in-memory data.

In [ ]:
# Enter the destination catalogue and schema, then build the full table name.
catalog = "YOUR_CATALOG"
schema = "YOUR_SCHEMA"
table = f"{catalog}.{schema}.{dataset_key}"

# Convert the cleaned data to a Spark table and store dates without a time component.
sdf = spark.createDataFrame(df)
sdf = sdf.withColumn("date", sdf["date"].cast("date"))

# Create the Delta table or overwrite its existing data.
sdf.write.format("delta").mode("overwrite").saveAsTable(table)

# Collect the dataset, filenames, download URL, release month and retrieval time into one source record.
sources = pd.DataFrame(
    [(dataset, baseline_file, download_file, source_url, download_month, retrieved_at, csv_file.name)],
    columns=["dataset", "baseline_file", "download_file", "source_url", "reference_period", "retrieved_at_utc", "output_file"],
)
# Save sources.csv after publication to mark this release as completed for the next run.
sources.to_csv(run / "sources.csv", index=False, date_format="%Y-%m-%d")

# Display the published row count, destination table and source record.
print(f"Published {len(df):,} rows to {table}")
display(sources)